# HW1: the Bradley-Terry model

**RLHF course · Pegah Alizadeh** · Due at session 2 · Reading: Chapter 5, "Training a Bradley-Terry Reward Model"

The reward models in RLHF are built on the Bradley-Terry model of paired comparisons. Before we fit one on language model outputs, we study it where **the truth is known**.

Item $i$ has a latent strength $\theta_i$, and

$$P(i \text{ beats } j) = \sigma(\theta_i - \theta_j), \qquad \sigma(u) = \frac{1}{1 + e^{-u}}.$$

| Level | What it asks | Who it's for |
|---|---|---|
| **Foundation** | Simulate data and fit the model | Everyone |
| **Core** | Build confidence intervals and test their coverage | The level we aim for |
| **Deep** | Derive the Fisher information, and study correlated comparisons | If you want to go further |

No GPU is needed. Write your answers in text cells in the notebook. Homework isn't graded, but it prepares you for the project.

**Sources.** The exercises were written for this course, using standard results:

- **The model:** Bradley and Terry (1952), *Biometrika* 39. The RLHF book, Chapter 5, explains how reward models use it.
- **Bradley-Terry as a logistic regression, and maximum likelihood:** A. Agresti, *Categorical Data Analysis*, in its chapter on paired comparisons. Also T. Truong, A. Haupt and S. Koyejo, [*Machine Learning from Human Preferences*](https://mlhp.stanford.edu), Section 2.3 and Exercise 1.17.6.
- **Identifiability:** *Machine Learning from Human Preferences*, Section 1.9.
- **Fisher information and the Wald interval:** any course on likelihood theory. *Machine Learning from Human Preferences* also uses Fisher information in Chapter 2.
- **Correlated comparisons and cluster-robust intervals:** K.-Y. Liang and S. Zeger (1986), *Biometrika* 73, and A. C. Cameron and D. Miller (2015), "A practitioner's guide to cluster-robust inference", *Journal of Human Resources* 50.

## The big picture

In RLHF, people compare two responses and say which is better. A **reward model** turns these comparisons into a score for
each response, and that score is what the model is later trained to increase. The statistical model behind it is
Bradley-Terry.

In this homework, the items are abstract (think of 6 responses, or 6 chess players), and we **choose the true strengths
ourselves**. Because we know the truth, we can check whether estimates and intervals behave as theory promises. This is the
standard way to test a statistical method before using it on real data.

### Key words

| Word | Meaning |
|---|---|
| **Latent strength** $\theta_i$ | An unobserved number for each item. Higher means more likely to win. |
| **Comparison** | One observation: items $i$ and $j$, and $y = 1$ if $i$ won |
| **Maximum likelihood** | The parameter values that make the observed comparisons most probable |
| **Fisher information** | How much the data tell us about the parameters. Its inverse approximates the variance of the estimate. |
| **Wald interval** | Estimate $\pm 1.96 \times$ standard error |
| **Coverage** | The fraction of repeated experiments in which the interval contains the true value |
| **Monte Carlo** | Estimating a quantity by repeating a random simulation many times |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def sigmoid(u):
    return 1 / (1 + np.exp(-u))


theta_true = np.array([0.0, 0.5, 1.0, -0.5, 1.5, -1.0])   # item 0 is the reference
K = len(theta_true)

### A first look: who beats whom?

Before simulating anything, compute the **true** probability that each item beats each other one. Row $i$, column $j$
is $P(i \text{ beats } j)$.

In [ ]:
P = sigmoid(theta_true[:, None] - theta_true[None, :])
print("strengths:", theta_true)
print(np.round(P, 2))

Three things to check in this table:

- **The diagonal is 0.5:** an item against itself is a coin flip.
- **$P(i \text{ beats } j) + P(j \text{ beats } i) = 1$:** the table is "antisymmetric" around 0.5.
- **Only differences matter:** item 4 ($\theta = 1.5$) beats item 2 ($\theta = 1.0$) with the same probability as item 1
  ($\theta = 0.5$) beats item 0 ($\theta = 0$), because both differences are 0.5.

---
# Foundation

### Question 1: identifiability

Show that $\theta$ and $\theta + c$ give the same likelihood for any constant $c$. What does this mean about what the data can tell us? This is why we fix $\theta_0 = 0$.

### Exercise 1: simulate comparisons

In [ ]:
def simulate(theta, n, rng):
    # TODO: return (i, j, y), three arrays of length n.
    # i, j: two different items, chosen uniformly at random
    # y: 1 if item i wins, with probability sigmoid(theta[i] - theta[j]), and 0 otherwise
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
i, j, y = simulate(theta_true, 50000, rng)
assert (i != j).all()
strongest = (i == 4) & (j == 5)      # theta difference of 2.5
assert abs(y[strongest].mean() - sigmoid(2.5)) < 0.03
print("Exercise 1: correct")

### Does the simulation match the model?

With 50,000 simulated comparisons, the observed win rate of each pair should be close to the true probability above.

In [ ]:
wins = np.zeros((K, K))
games = np.zeros((K, K))
np.add.at(games, (i, j), 1)
np.add.at(wins, (i, j), y)
observed = np.where(games > 0, wins / np.maximum(games, 1), np.nan)
print("Observed win rate of row item against column item:")
print(np.round(observed, 2))
print("Largest gap from the true probabilities:", np.round(np.nanmax(np.abs(observed - P)), 3))

Each pair is compared about 1,700 times here, so each observed rate has a standard error of about
$\sqrt{0.25 / 1700} \approx 0.012$. Gaps of a few hundredths are just sampling noise.

### Exercise 2: Bradley-Terry is a logistic regression

Writing $\beta = (\theta_1, \ldots, \theta_{K-1})$, each comparison is a row $x$ with $+1$ for item $i$ and $-1$ for item $j$, and $P(y = 1) = \sigma(x^\top \beta)$. There is no intercept.

**A tiny example with 3 items**, where item 0 is the reference and has no column:

| Comparison | Column for item 1 | Column for item 2 | $x^\top \beta$ |
|---|---|---|---|
| 1 against 2 | $+1$ | $-1$ | $\theta_1 - \theta_2$ |
| 2 against 0 | $0$ | $+1$ | $\theta_2 - \theta_0 = \theta_2$ |
| 0 against 1 | $-1$ | $0$ | $\theta_0 - \theta_1 = -\theta_1$ |

Each row gives exactly the difference that Bradley-Terry needs. So fitting Bradley-Terry **is** fitting a logistic
regression on this design matrix, and any logistic regression software can do it.

**Newton's method** improves the estimate using the curvature of the log-likelihood. For logistic regression, the
log-likelihood is concave, so Newton's method converges quickly to the unique maximum, usually in a handful of steps.
This algorithm is also called **iteratively reweighted least squares**.

Complete `design_matrix`, then fit the model with Newton's method. The update is

$$\beta \leftarrow \beta + \left(X^\top W X\right)^{-1} X^\top (y - p), \qquad W = \mathrm{diag}\big(p_m (1 - p_m)\big).$$

In [ ]:
def design_matrix(i, j, K):
    # TODO: return an (n, K-1) matrix. Row m has +1 in the column of item i[m] and -1 in the
    # column of item j[m], and item 0 has no column, because theta[0] = 0 is fixed.
    raise NotImplementedError

In [ ]:
def fit(X, y, steps=25):
    # TODO: return the maximum likelihood estimate of beta (theta without item 0), using Newton's method.
    # A small ridge, such as 1e-6 * np.eye(d), keeps the matrix invertible on unlucky samples.
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
i, j, y = simulate(theta_true, 20000, rng)
X = design_matrix(i, j, K)
assert X.shape == (20000, K - 1)
beta_hat = fit(X, y)
print("True:     ", theta_true[1:])
print("Estimated:", np.round(beta_hat, 2))
assert np.abs(beta_hat - theta_true[1:]).max() < 0.1
print("Exercise 2: correct")

### Watch Newton's method converge

Here are the estimates after 1, 2, 3, 4 and 5 Newton steps, on the same data:

In [ ]:
print("Your design matrix for 3 comparisons (1 vs 2, 2 vs 0, 0 vs 1), with K = 3:")
print(design_matrix(np.array([1, 2, 0]), np.array([2, 0, 1]), 3))
print()
for steps in range(1, 6):
    print(f"after {steps} steps:", np.round(fit(X, y, steps=steps), 4))
print("true values:  ", theta_true[1:])

Newton's method usually stops changing after 3 or 4 steps. The estimates are close to the truth but not equal to it:
that remaining gap is sampling error, which the next exercises measure.

---
# Core

### Exercise 3: Wald confidence intervals

Asymptotically, $\hat\beta \approx \mathcal{N}\big(\beta, I(\beta)^{-1}\big)$, with Fisher information $I(\beta) = X^\top W X$. Build 95% intervals for each $\beta_k$.

**Where this comes from.** With many independent comparisons, the maximum likelihood estimate is approximately normal,
centered on the truth. Its covariance is the inverse Fisher information, evaluated at the estimate. So:

1. Compute $I(\hat\beta) = X^\top W X$, with $W$ built from the fitted probabilities.
2. Invert it. The **standard error** of $\hat\beta_k$ is the square root of the $k$-th diagonal entry.
3. The interval is $\hat\beta_k \pm 1.96 \times \text{standard error}$.

The Fisher information is the same matrix that appears in your Newton step: the curvature of the log-likelihood.

In [ ]:
def wald_interval(X, beta, z=1.96):
    # TODO: return (lower, upper), arrays of length d, from the inverse Fisher information.
    raise NotImplementedError

### Your intervals on one dataset

With 500 comparisons, here are the estimates, the intervals, and whether each interval contains the true value:

In [ ]:
rng = np.random.default_rng(13)
i, j, y = simulate(theta_true, 500, rng)
X = design_matrix(i, j, K)
beta_hat = fit(X, y)
lower, upper = wald_interval(X, beta_hat)
assert (lower < upper).all(), "Each lower bound should be below its upper bound."
for k in range(K - 1):
    covered = lower[k] <= theta_true[k + 1] <= upper[k]
    print(f"item {k + 1}: estimate {beta_hat[k]:6.2f}, interval [{lower[k]:6.2f}, {upper[k]:6.2f}], "
          f"true {theta_true[k + 1]:5.2f}, covered: {covered}")

On one dataset, most intervals should contain the truth, but perhaps not all. A 95% interval is allowed to miss 5% of
the time. Misses can also come together: every estimate is measured relative to item 0, so if item 0 happens to do
unusually well or badly in the data, all the estimates shift at once. Whether it really misses only 5% of the time can't be seen from one dataset: that's what coverage measures.

### Exercise 4: do they really cover 95%?

A confidence level is a claim about repeated sampling, and in simulation we can test it. Estimate the coverage for several sample sizes, then plot it.

**The recipe:** repeat many times: simulate a new dataset, fit the model, build the intervals, and record whether each
contains the true value. The fraction of hits estimates the coverage. If the method is right, it should be close to 0.95.

**Question 2.** How close to 95% is the coverage at small $n$? How many simulations do you need before differences from 95% are more than simulation noise? (Compute the Monte Carlo standard error.)

In [ ]:
def coverage(theta, n, sims, rng):
    # TODO: over `sims` simulated datasets, return the fraction of intervals containing the truth,
    # as an array with one entry per item (excluding item 0).
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(2)
sizes = [100, 300, 1000]
results = [coverage(theta_true, n, 300, rng) for n in sizes]
for n, c in zip(sizes, results):
    print(f"n = {n:5d}: mean coverage {c.mean():.3f}, by item {np.round(c, 2)}")
plt.plot(sizes, [c.mean() for c in results], marker="o")
plt.axhline(0.95, linestyle="--", color="gray")
plt.xscale("log")
plt.xlabel("Number of comparisons")
plt.ylabel("Mean coverage")
plt.show()

**How to read the plot.** The dashed line is the nominal 95%. Points above it mean intervals that are a little too wide,
and points below it mean intervals that are too narrow, which is the more dangerous error. Remember that each point is
itself an estimate from a finite number of simulations, so small differences from 0.95 may be noise: Question 2 asks
how small.

---
# Deep

### Question 3: Fisher information

Derive the score and the Fisher information for one comparison, and show that $I(\beta) = \sum_m p_m(1-p_m)\, x_m x_m^\top$.

Then answer: (a) why is $I(\beta)$ singular if we don't fix $\theta_0$? (b) Which comparisons carry the most information? (c) What does that suggest about how to choose which pairs to label?

### Exercise 5: correlated comparisons

Our intervals assume independent comparisons. In real preference data, one annotator gives many labels and annotators differ. Chapter 5 raises a related concern about correlated comparisons.

Simulate this. Each annotator has strengths $\theta + u$, with $u_k \sim \mathcal{N}(0, \tau^2)$ independently per item. Estimate the coverage of the same Wald interval, and of an interval you propose that accounts for annotators.

**Target:** with annotator effects, $\hat\beta$ estimates a population-averaged effect, which is not exactly $\theta_k - \theta_0$. The cell below gives you a reasonable target: `truth(theta)`, the population-averaged log-odds against item 0, attenuated as in a marginal logistic model.

**Why correlation matters.** Labels from the same annotator share that annotator's tastes, so they aren't independent
pieces of information. Ten labels from one person tell us about that person, not ten times as much about the population.

<details><summary>Hint: the design effect</summary>

For an average of $n$ observations in groups of $m$, with correlation $\rho$ between observations of the same group,
the variance is inflated by about $1 + (m - 1)\rho$, compared with independent data. This factor is called the
**design effect**, and $n / (1 + (m - 1)\rho)$ is the **effective sample size**.
</details>

<details><summary>Hint: an interval that accounts for annotators</summary>

A **cluster-robust (sandwich) variance** keeps the inverse Fisher information as the "bread", but replaces the "meat" with
the sum over annotators of $s_a s_a^\top$, where $s_a$ is the total score (gradient of the log-likelihood) of annotator
$a$'s labels. An alternative is a **bootstrap that resamples whole annotators**.
</details>

**Question 4.** How does coverage change with $\tau$, and with the number of labels per annotator for a fixed total? What is the effective sample size when an annotator gives many labels? Why does this matter for the class labels we'll collect in session 2?

In [ ]:
def truth(theta, tau=None, draws=200000, seed=123):
    """The population-averaged log-odds of each item against item 0, found by Monte Carlo."""
    tau = TAU if tau is None else tau
    noise = np.random.default_rng(seed).normal(0, tau, (draws, 1)) * np.sqrt(2)
    p = sigmoid(theta[1:] - theta[0] + noise).mean(axis=0)
    return np.log(p / (1 - p))


TAU = 1.0
print("Target:", np.round(truth(theta_true), 3))

In [ ]:
def simulate_annotators(theta, n_annotators, per_annotator, tau, rng):
    # TODO: each annotator has their own strengths: theta + u, with u ~ Normal(0, tau^2) for each item.
    # They make per_annotator comparisons. Return (i, j, y, annotator), each of the total length.
    raise NotImplementedError


def coverage_correlated(theta, n_annotators, per_annotator, tau, sims, rng):
    # TODO: as coverage(), using simulate_annotators and the same Wald interval.
    raise NotImplementedError


def cluster_interval(X, y, beta, annotator, z=1.96):
    # TODO: propose an interval that accounts for the annotators, for example a cluster-robust
    # sandwich estimator or a bootstrap that resamples whole annotators.
    raise NotImplementedError


def coverage_cluster(theta, n_annotators, per_annotator, tau, sims, rng):
    # TODO: coverage of your proposed interval.
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(3)
for n_annotators, per_annotator in [(100, 10), (20, 50), (10, 100)]:
    naive = coverage_correlated(theta_true, n_annotators, per_annotator, TAU, 200, rng)
    robust = coverage_cluster(theta_true, n_annotators, per_annotator, TAU, 200, rng)
    print(f"{n_annotators:3d} annotators x {per_annotator:3d} labels: Wald {naive.mean():.3f}, cluster-robust {robust.mean():.3f}")

## What you learned

- **Bradley-Terry** gives each item a strength, and only **differences** between strengths are identified
- **Fitting it is a logistic regression** on a design matrix with $+1$ and $-1$, solved by Newton's method
- **The Fisher information** gives standard errors, and **Wald intervals** that cover about 95% when comparisons are independent
- **Coverage can be tested by simulation,** as long as you account for Monte Carlo error
- **Correlated comparisons,** such as many labels from one annotator, make naive intervals too narrow. This matters for
  real preference data, and for the labels we collect in session 2.

**Going further (optional):** use `truth` with care. Is the target exactly right when comparisons don't involve item 0? Try a mixed-effects model, or study how many annotators the sandwich estimator needs.